# Capa Gold — KPIs por Vendedor y Mes

Agrega el rendimiento de cada vendedor a nivel **vendedor + tienda + mes**, con KPIs de volumen, ingresos y descuentos. Permite ranking de vendedores, comparativas entre tiendas y seguimiento de tendencias en Power BI.

In [0]:
# Limpieza de widgets previos al inicio del proceso para evitar conflictos con ejecuciones anteriores
dbutils.widgets.removeAll()

In [0]:
# Importación de librerías de PySpark para transformaciones y funciones con alias explícito (F)
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
# Parámetros de entrada: catálogo, esquema origen (silver) y destino (golden)
dbutils.widgets.text("catalogo", "cat_ftr_smartdata_dev")
dbutils.widgets.text("source", "silver")
dbutils.widgets.text("sink", "golden")

In [0]:
# Recuperación de los valores de los widgets para usarlos como variables en el proceso
catalogo = dbutils.widgets.get("catalogo")
schema_source = dbutils.widgets.get("source")
schema_sink = dbutils.widgets.get("sink")

In [0]:
# Lectura de la tabla Silver enriquecida como fuente de la capa Gold
# ANIO, MES, TRIMESTRE, IMPORTE_BRUTO e IMPORTE_NETO ya están pre-calculados en Silver
df_silver = spark.table(f"{catalogo}.{schema_source}.ventas_productos_categorias")
#display(df_silver.limit(5))

In [0]:
# Agregación mensual por vendedor y tienda usando importes pre-calculados de Silver
# Permite rankings de vendedores, comparativas entre tiendas y detección de outliers en Power BI
df_kpi_vendedor = (
    df_silver
    .groupBy(
        F.col("ID_VENDEDOR"),
        F.col("TIENDA"),
        F.col("ANIO"),
        F.col("MES"),
        F.col("TRIMESTRE")
    )
    .agg(
        # — Volumen —
        F.countDistinct("ID_PEDIDO").alias("NUM_PEDIDOS"),
        F.countDistinct("ID_ARTICULO").alias("NUM_ARTICULOS_DISTINTOS"),
        F.sum("CANTIDAD").alias("TOTAL_UNIDADES"),
        F.count("*").alias("NUM_LINEAS"),

        # — Ingresos (campos pre-calculados en Silver) —
        F.round(F.sum("IMPORTE_BRUTO"), 2).alias("INGRESO_BRUTO"),
        F.round(F.sum("IMPORTE_NETO"),  2).alias("INGRESO_NETO"),
        F.round(F.sum("IMPORTE_BRUTO") - F.sum("IMPORTE_NETO"), 2).alias("INGRESO_DESCUENTO"),

        # — Descuento —
        F.count(F.when(F.col("DESCUENTO") > 0, 1)).alias("NUM_LINEAS_CON_DESCUENTO"),
        F.round(
            F.avg(F.when(F.col("DESCUENTO") > 0, F.col("DESCUENTO"))), 2
        ).alias("DESCUENTO_MEDIO_PCT"),

        # — Mix de tipo de venta —
        F.count(F.when(F.col("ETIQUETA_DESCUENTO") == "Precio Normal",    1)).alias("NUM_PRECIO_NORMAL"),
        F.count(F.when(F.col("ETIQUETA_DESCUENTO") == "Promoción",        1)).alias("NUM_PROMOCION"),
        F.count(F.when(F.col("ETIQUETA_DESCUENTO") == "Bono Empresarial", 1)).alias("NUM_BONO_EMPRESARIAL"),
        F.count(F.when(F.col("ETIQUETA_DESCUENTO") == "Regalo",           1)).alias("NUM_REGALO"),
    )
    .withColumn(
        "PCT_LINEAS_CON_DESCUENTO",
        F.round(F.col("NUM_LINEAS_CON_DESCUENTO") / F.col("NUM_LINEAS") * 100, 2)
    )
    .withColumn(
        "TICKET_PROMEDIO",
        F.round(F.col("INGRESO_NETO") / F.col("NUM_PEDIDOS"), 2)
    )
    # — Auditoría Gold —
    .withColumn("FECHA_ACTUALIZACION", F.current_timestamp())  # Cuándo se calculó la agregación
    .withColumn("INGESTION_USER",     F.current_user())        # Quién ejecutó el pipeline Gold
)

#display(df_kpi_vendedor.orderBy("TIENDA", "ANIO", "MES", "ID_VENDEDOR").limit(10))

In [0]:
# Escritura en Gold con overwriteSchema para soportar la evolución del esquema
target_table = f"{catalogo}.{schema_sink}.kpi_vendedor_mes"

(
    df_kpi_vendedor
    .select(
        F.col("ID_VENDEDOR").cast("string"),
        F.col("TIENDA").cast("string"),
        F.col("ANIO").cast("int"),
        F.col("MES").cast("int"),
        F.col("TRIMESTRE").cast("int"),
        F.col("NUM_PEDIDOS").cast("int"),
        F.col("NUM_ARTICULOS_DISTINTOS").cast("int"),
        F.col("TOTAL_UNIDADES").cast("int"),
        F.col("NUM_LINEAS").cast("int"),
        F.col("INGRESO_BRUTO").cast("double"),
        F.col("INGRESO_NETO").cast("double"),
        F.col("INGRESO_DESCUENTO").cast("double"),
        F.col("DESCUENTO_MEDIO_PCT").cast("double"),
        F.col("NUM_LINEAS_CON_DESCUENTO").cast("int"),
        F.col("PCT_LINEAS_CON_DESCUENTO").cast("double"),
        F.col("TICKET_PROMEDIO").cast("double"),
        F.col("NUM_PRECIO_NORMAL").cast("int"),
        F.col("NUM_PROMOCION").cast("int"),
        F.col("NUM_BONO_EMPRESARIAL").cast("int"),
        F.col("NUM_REGALO").cast("int"),
        F.col("FECHA_ACTUALIZACION").cast("timestamp"),
        F.col("INGESTION_USER").cast("string"),
    )
    .coalesce(4)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"✓ Tabla '{target_table}' escrita correctamente en capa Gold.")

In [0]:
%sql
-- Verificación de kpi_vendedor_mes: top vendedores por ingreso neto en el último mes disponible
SELECT *
FROM cat_ftr_smartdata_dev.golden.kpi_vendedor_mes
ORDER BY ANIO DESC, MES DESC, INGRESO_NETO DESC
LIMIT 10